In [ ]:
# @title
import io, base64, cv2, numpy as np
from PIL import Image
from IPython.display import HTML

# 1) Recomponer: overlay del backend encima de cada frame original
out = cv2.VideoWriter("/content/_tmp.mp4", cv2.VideoWriter_fourcc(*"mp4v"),
                      20, (result["width"], result["height"]))
for fr in result["frames"]:
    base = Image.open(frame_files[fr["frame"]]).convert("RGBA")
    ov = Image.open(io.BytesIO(base64.b64decode(fr["overlay_png"]))).convert("RGBA")
    comp = Image.alpha_composite(base, ov).convert("RGB")
    out.write(cv2.cvtColor(np.array(comp), cv2.COLOR_RGB2BGR))
out.release()

# 2) Re-encodear a H.264 para que el navegador lo reproduzca
!ffmpeg -y -i /content/_tmp.mp4 -vcodec libx264 -pix_fmt yuv420p /content/resultado.mp4 -loglevel quiet
print("✅ video listo")

# 3) Reproducir en el notebook
mp4 = open("/content/resultado.mp4", "rb").read()
HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{base64.b64encode(mp4).decode()}" type="video/mp4"></video>')

✅ video listo


#INSTALACIÓN DE DEPENDENCIAS Y NÚCLEO SAM 3

In [ ]:
import os
import sys

print("Instalando librerías base del servidor...")
!pip install -q fastapi "uvicorn[standard]" pyngrok nest_asyncio deep-translator websockets scipy torch torchvision pillow opencv-python

print("Clonando repositorio oficial de Meta AI SAM 3...")
%cd /content
if not os.path.exists("sam3"):
    !git clone https://github.com/facebookresearch/sam3.git

# Añadir las rutas al sistema de Python para evitar errores de importación
%cd /content/sam3
!pip install -q -e .
sys.path.insert(0, "/content/sam3")
%cd /content

print("\n¡Entorno de ejecución preparado exitosamente!")

/content/sam3/sam3/model/model_misc.py:70: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


config.json:   0%|          | 0.00/25.8k [00:00<?, ?B/s]

sam3.pt:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

✅ Modelo cargado
🌍 Backend en: https://bulk-dismiss-exclaim.ngrok-free.dev
✅ Servidor encendido


# Dependencias del servidor

In [ ]:
!pip install -q fastapi "uvicorn[standard]" pyngrok nest_asyncio
!pip install -q deep-translator

# --- Librería SAM 3 + realineo de numpy ---
%cd /content
![ -d sam3 ] || git clone https://github.com/facebookresearch/sam3.git
%cd /content/sam3
!pip install -q -e .
!pip install -q --force-reinstall --no-deps "numpy>=2,<2.6"


# --- Tu backend (tu código del repo) ---
%cd /content
![ -d sam3-backend ] || git clone https://github.com/nigrumanimam6-cmd/sam3-backend.git
!cd /content/sam3-backend && git pull

/content
/content/sam3
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sam3 (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1

# Creación del Backend

In [ ]:
%%writefile main.py
import io
import os
import json
import base64
import asyncio
import tempfile
from collections import defaultdict

import numpy as np
import torch
import cv2
from PIL import Image
from scipy.optimize import linear_sum_assignment
from fastapi import FastAPI, WebSocket, WebSocketDisconnect

# --- Estado global: el modelo de imagen (compartido) ------------------------
_processor = None

def load_model():
    """Carga SAM 3 (modelo de imagen). Llamar UNA vez al arrancar."""
    global _processor
    from sam3.model_builder import build_sam3_image_model
    from sam3.model.sam3_image_processor import Sam3Processor
    model = build_sam3_image_model()
    _processor = Sam3Processor(model)
    return _processor

# ============================================================================
#  1) SEGMENTACION DE IMAGEN (overlay cian)
# ============================================================================
def segmentar(image, prompt):
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        state = _processor.set_image(image)
        out = _processor.set_text_prompt(state=state, prompt=prompt)
    masks, scores = out["masks"], out["scores"]
    n = int(masks.shape[0])
    if n == 0:
        return None, 0, 0.0
    m = masks[0, 0].float().cpu().numpy()
    thr = 0.5 if m.max() <= 1.0 + 1e-3 else 0.0
    m_bin = m > thr
    H, W = m_bin.shape
    rgba = np.zeros((H, W, 4), dtype=np.uint8)
    rgba[m_bin] = [0, 200, 255, 130]
    buf = io.BytesIO()
    Image.fromarray(rgba).save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("ascii"), n, float(scores[0].float().cpu())

# ============================================================================
#  2) MOTOR DE VIDEO (frame-por-frame + rastreador)
# ============================================================================
MIN_SCORE = 0.5
OBJ_COLORS = [(0, 200, 255), (255, 90, 90), (120, 220, 120), (255, 200, 60),
              (200, 120, 255), (255, 140, 200), (120, 200, 255), (180, 180, 120)]

def _png_b64(rgba):
    buf = io.BytesIO()
    Image.fromarray(rgba).save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("ascii")

def _iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def _cdist(c1, c2):
    return ((c1[0] - c2[0]) ** 2 + (c1[1] - c2[1]) ** 2) ** 0.5

def _color_hist(img, mask, bins=4):
    pix = img[mask]
    if len(pix) == 0:
        return np.zeros(bins ** 3, np.float32)
    q = (pix.astype(np.int32) * bins // 256).clip(0, bins - 1)
    idx = q[:, 0] * bins * bins + q[:, 1] * bins + q[:, 2]
    h = np.bincount(idx, minlength=bins ** 3).astype(np.float32)
    return h / (h.sum() + 1e-6)

def _hist_sim(h1, h2):
    return float(np.minimum(h1, h2).sum())

def _translate(text):
    try:
        from deep_translator import GoogleTranslator
        return GoogleTranslator(source="auto", target="en").translate(text) or text
    except Exception:
        return text

def _detect(image_pil, prompt):
    img = np.array(image_pil)
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        state = _processor.set_image(image_pil)
        out = _processor.set_text_prompt(state=state, prompt=prompt)
    masks, scores = out["masks"], out["scores"]
    dets = []
    for i in range(int(masks.shape[0])):
        sc = float(scores[i].float().cpu())
        if sc < MIN_SCORE:
            continue
        m = masks[i, 0].float().cpu().numpy()
        mb = m > (0.5 if m.max() <= 1.0 + 1e-3 else 0.0)
        if mb.sum() < 200:
            continue
        ys, xs = np.where(mb)
        dets.append({
            "mask": mb,
            "centroid": (float(xs.mean()), float(ys.mean())),
            "bbox": (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())),
            "score": sc,
            "hist": _color_hist(img, mb),
            "label": prompt,
        })
    return dets

class Tracker:
    def __init__(self, w_img, match_thr=0.6, max_lost=30, w_motion=0.6):
        self.tracks, self.next_id = {}, 0
        self.w_img = w_img
        self.match_thr, self.max_lost, self.w_motion = match_thr, max_lost, w_motion

    def _predict(self, t):
        tr = t["trail"]
        if len(tr) >= 2:
            (x0, y0), (x1, y1) = tr[-2], tr[-1]
            return (x1 + (x1 - x0), y1 + (y1 - y0))
        return tr[-1]

    def update(self, dets):
        ids = list(self.tracks.keys())
        assign = {}
        if ids and dets:
            cost = np.ones((len(dets), len(ids)))
            for i, d in enumerate(dets):
                for j, tid in enumerate(ids):
                    t = self.tracks[tid]
                    motion = min(1 - _iou(d["bbox"], t["bbox"]),
                                 _cdist(d["centroid"], self._predict(t)) / (0.25 * self.w_img))
                    app = 1 - _hist_sim(d["hist"], t["hist"])
                    cost[i, j] = self.w_motion * motion + (1 - self.w_motion) * app
            for r, c in zip(*linear_sum_assignment(cost)):
                if cost[r, c] <= self.match_thr:
                    assign[r] = ids[c]
        seen, out = set(), []
        for i, d in enumerate(dets):
            if i in assign:
                tid = assign[i]
                t = self.tracks[tid]
                t.update(bbox=d["bbox"], centroid=d["centroid"], lost=0)
                t["trail"].append(d["centroid"])
                t["hist"] = 0.7 * t["hist"] + 0.3 * d["hist"]
            else:
                tid = self.next_id
                self.next_id += 1
                self.tracks[tid] = {"bbox": d["bbox"], "centroid": d["centroid"],
                                    "trail": [d["centroid"]], "lost": 0, "hist": d["hist"]}
            seen.add(tid)
            out.append((tid, d, list(self.tracks[tid]["trail"])))
        for tid in ids:
            if tid not in seen:
                self.tracks[tid]["lost"] += 1
        for tid in [t for t in self.tracks if self.tracks[t]["lost"] > self.max_lost]:
            del self.tracks[tid]
        return out

def _extract_frames(video_bytes, max_frames=None, stride=1, target_fps=None, max_seconds=None):
    path = tempfile.mktemp(suffix=".mp4")
    with open(path, "wb") as f:
        f.write(video_bytes)
    cap = cv2.VideoCapture(path)
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    if target_fps:
        stride = max(1, round(src_fps / float(target_fps)))
    max_src = int(max_seconds * src_fps) if max_seconds else None
    frames, i = [], 0
    while True:
        ret, fr = cap.read()
        if not ret:
            break
        if max_src is not None and i >= max_src:
            break
        if i % max(1, stride) == 0:
            frames.append(Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))
            if max_frames and len(frames) >= max_frames:
                break
        i += 1
    cap.release()
    try:
        os.remove(path)
    except OSError:
        pass
    eff_fps = round(src_fps / max(1, stride), 2)
    return frames, eff_fps

def process_video(frames, prompt, min_presence=0.25, export_path=None, fps_out=10):
    N = len(frames)
    W, H = frames[0].size
    tracker = Tracker(w_img=W)
    per_frame, obj_frames = [], defaultdict(list)

    originals = [p.strip() for p in prompt.split(",") if p.strip()] or [prompt]
    english = [_translate(p) for p in originals]
    yield {"type": "prompts", "pairs": [[o, e] for o, e in zip(originals, english)]}

    for fi, img in enumerate(frames):
        dets = []
        for orig, eng in zip(originals, english):
            for d in _detect(img, eng):
                d["label"] = orig
                dets.append(d)
        tracked = tracker.update(dets)
        per_frame.append(tracked)
        for tid, d, _ in tracked:
            obj_frames[tid].append((fi, d))
        yield {"type": "progress", "done": fi + 1, "total": N}

    stable = sorted([t for t, fs in obj_frames.items() if len(fs) >= min_presence * N])
    color_of = {t: OBJ_COLORS[i % len(OBJ_COLORS)] for i, t in enumerate(stable)}

    objects = []
    for tid in stable:
        fi, d = max(obj_frames[tid], key=lambda x: x[1]["mask"].sum())
        img = np.array(frames[fi])
        m = d["mask"]
        ys, xs = np.where(m)
        cut = img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        cm = m[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        rgba = np.zeros((cut.shape[0], cut.shape[1], 4), np.uint8)
        rgba[..., :3] = cut
        rgba[..., 3] = np.where(cm, 255, 0)
        objects.append({
            "id": int(tid),
            "color": list(color_of[tid]),
            "label": d.get("label", prompt),
            "trail": [[int(fi), float(dd["centroid"][0]), float(dd["centroid"][1])]
                      for fi, dd in obj_frames[tid]],
            "thumb_png": _png_b64(rgba),
        })

    yield {"type": "result_meta", "n_frames": N, "width": int(W), "height": int(H), "objects": objects}

    writer = None
    if export_path:
        writer = cv2.VideoWriter(export_path, cv2.VideoWriter_fourcc(*"mp4v"), float(fps_out) if fps_out else 10.0, (W, H))

    sset = set(stable)
    for fi, tracked in enumerate(per_frame):
        ov = np.zeros((H, W, 4), np.uint8)
        objs = []
        for tid, d, _ in tracked:
            if tid in sset:
                r, g, b = color_of[tid]
                ov[d["mask"]] = [r, g, b, 130]
                objs.append({
                    "id": int(tid),
                    "bbox": [int(x) for x in d["bbox"]],
                })

        base = np.array(frames[fi]).astype(np.float32)
        a = ov[..., 3:4].astype(np.float32) / 255.0
        comp = (base * (1 - a) + ov[..., :3] * a).astype(np.uint8)

        jb = io.BytesIO()
        Image.fromarray(comp).save(jb, format="JPEG", quality=80)
        comp_jpg = base64.b64encode(jb.getvalue()).decode("ascii")
        yield {"type": "frame", "frame": fi, "comp_jpg": comp_jpg, "objects": objs}

        if writer is not None:
            writer.write(cv2.cvtColor(comp, cv2.COLOR_RGB2BGR))

    if writer is not None:
        writer.release()

    yield {"type": "result_done"}

app = FastAPI(title="SAM3 Backend")

@app.get("/")
def health():
    return {"status": "ok"}

async def _run_video(websocket, video_bytes, prompt, mp, max_frames, stride, target_fps, max_seconds=None):
    frames, eff_fps = _extract_frames(video_bytes, max_frames=max_frames, stride=stride, target_fps=target_fps, max_seconds=max_seconds)
    if not frames:
        await websocket.send_text(json.dumps({"type": "error", "msg": "no se pudieron leer frames"}))
        return None

    await websocket.send_text(json.dumps({"type": "video_info", "n_frames": len(frames), "proc_fps": eff_fps}))
    export_path = tempfile.mktemp(suffix=".mp4")

    loop = asyncio.get_running_loop()
    queue = asyncio.Queue()

    def _worker():
        try:
            for update in process_video(frames, prompt, mp, export_path=export_path, fps_out=eff_fps):
                loop.call_soon_threadsafe(queue.put_nowait, update)
        except Exception as e:
            loop.call_soon_threadsafe(queue.put_nowait, {"type": "error", "msg": str(e)})
        finally:
            loop.call_soon_threadsafe(queue.put_nowait, None)

    loop.run_in_executor(None, _worker)
    while True:
        update = await queue.get()
        if update is None:
            break
        if update.get("type") == "result_meta":
            update["proc_fps"] = eff_fps
        await websocket.send_text(json.dumps(update))

    return export_path

@app.websocket("/ws")
async def ws(websocket: WebSocket):
    await websocket.accept()
    current_image = None
    video_chunks = []
    video_meta = {}
    last_export = None
    try:
        while True:
            msg = json.loads(await websocket.receive_text())
            t = msg.get("type")

            if t == "image":
                raw = base64.b64decode(msg["data"])
                current_image = Image.open(io.BytesIO(raw)).convert("RGB")
                W, H = current_image.size
                await websocket.send_text(json.dumps({"type": "image_ready", "width": W, "height": H}))

            elif t == "text":
                if current_image is None:
                    await websocket.send_text(json.dumps({"type": "error", "msg": "primero carga una imagen"}))
                    continue
                b64, n, score = segmentar(current_image, msg.get("prompt", ""))
                W, H = current_image.size
                await websocket.send_text(json.dumps({
                    "type": "mask", "prompt": msg.get("prompt", ""),
                    "n": n, "score": round(score, 4), "width": W, "height": H, "png": b64}))

            elif t == "video":
                raw = base64.b64decode(msg["data"])
                last_export = await _run_video(websocket, raw, msg.get("prompt", ""), float(msg.get("min_presence", 0.25)), msg.get("max_frames"), int(msg.get("stride", 1)), msg.get("target_fps"), msg.get("max_seconds"))

            elif t == "video_start":
                video_chunks = []
                video_meta = {
                    "prompt": msg.get("prompt", ""),
                    "min_presence": float(msg.get("min_presence", 0.25)),
                    "max_frames": msg.get("max_frames"),
                    "stride": int(msg.get("stride", 1)),
                    "target_fps": msg.get("target_fps"),
                    "max_seconds": msg.get("max_seconds"),
                }
                await websocket.send_text(json.dumps({"type": "video_ack"}))

            elif t == "video_chunk":
                video_chunks.append(msg["data"])

            elif t == "video_end":
                raw = base64.b64decode("".join(video_chunks))
                video_chunks = []
                last_export = await _run_video(websocket, raw, video_meta.get("prompt", ""), video_meta.get("min_presence", 0.25), video_meta.get("max_frames"), video_meta.get("stride", 1), video_meta.get("target_fps"), video_meta.get("max_seconds"))

            elif t == "export_video":
                if not last_export or not os.path.exists(last_export):
                    await websocket.send_text(json.dumps({"type": "error", "msg": "procesa un video primero"}))
                    continue
                with open(last_export, "rb") as f:
                    b64 = base64.b64encode(f.read()).decode("ascii")
                await websocket.send_text(json.dumps({"type": "export_start", "size": len(b64)}))
                CH = 256 * 1024
                for i in range(0, len(b64), CH):
                    await websocket.send_text(json.dumps({"type": "export_chunk", "data": b64[i:i + CH]}))
                await websocket.send_text(json.dumps({"type": "export_end"}))
            else:
                await websocket.send_text(json.dumps({"type": "error", "msg": "tipo no soportado"}))
    except WebSocketDisconnect:
        print("Cliente desconectado")

# Inicialización del Servidor y Enlace por Túnel

In [ ]:
# ============================================================================
# 3. CARGA DE MODELO Y ACTIVACIÓN DEL TÚNEL DE COMUNICACIÓN EN VIVO
# ============================================================================
import sys
import nest_asyncio
import uvicorn
import threading
import time
from pyngrok import ngrok
from google.colab import userdata

# Mapear rutas internas
sys.path.insert(0, "/content/sam3")
sys.path.insert(0, "/content")

# Importar el módulo main.py generado en la celda anterior
import main

print("Cargando pesos fundacionales de SAM 3 en la GPU...")
main.load_model()
print("¡Modelo inicializado con éxito en los núcleos tensores gráficos!")

# Activar la tunelización con Ngrok a través de las variables secretas
nest_asyncio.apply()
token_ngrok = userdata.get("NGROK_AUTHTOKEN")
ngrok.set_auth_token(token_ngrok)
ngrok.kill()

# Forzar la generación del enlace público seguro
url_publica = ngrok.connect(8000).public_url
url_websocket = url_publica.replace("http://", "ws://").replace("https://", "ws://") + "/ws"

print("\n" + "="*60)
print(f" DIRECCIÓN WEBSOCKET PARA TU CLIENTE FLUTTER (DART):")
print(f"{url_websocket}")
print("="*60 + "\n")

# Ejecutar el servidor asíncrono Uvicorn en un hilo independiente (Daemon)
def encender_servidor():
    uvicorn.run(main.app, host="0.0.0.0", port=8000, log_level="warning", ws_max_size=64 * 1024 * 1024)

threading.Thread(target=encender_servidor, daemon=True).start()
time.sleep(3)
print("[SERVIDOR ACTIVO, ESCUCHANDO PETICIONES MULTIMEDIA]")